# Task 2.2 Reproduction of Contribution: Random Fourier Features

*   **Contribution attempting to reproduce:** I am successfully explicitly reproducing the "Random Fourier Features" extraction strategy for approximating the Gaussian (RBF) Kernel as outlined cleanly in **Algorithm 1**, and then applying a fast linear Ridge Classifier over the explicit constructed feature space $Z$.
*   **Evaluation Metric:** Accuracy (percentage of samples correctly classified).


In [1]:
import numpy as np
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Re-load for standalone execution consistency
df = pd.read_csv('data/dataset.csv')
X = StandardScaler().fit_transform(df[['feature_1', 'feature_2']].values)
y = df['label'].values

# --------------------------------
# HYPERPARAMETERS SET IN ONE PLACE
# --------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
D_FEATURES = 500  # Number of random features D
GAMMA = 1.0       # Gaussian Kernel bandwidth parameter


/var/folders/yv/v55sx5nd7lj74s_w9kvkdk_h0000gn/T/ipykernel_4628/1984279719.py:5: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### Step 1: Draw Random Directions and Shift Constants
This code block explicitly generates the random projection weights $\omega_1, \dots, \omega_D$ from the Gaussian kernel's scaled probability distribution $p(\omega) = \mathcal{N}(0, 2\gamma)$ and spatial shifts $b_1, \dots, b_D$ from a Uniform distribution over $[0, 2\pi]$. This code chunk corresponds exactly to the random sampling matrix described in **Algorithm 1**, utilizing Bochner's theorem parameters formulated for the Gaussian kernel.

In [2]:
# Standard deviation of the normal projection distribution depends on 2*gamma.
std_dev = np.sqrt(2 * GAMMA)
W = np.random.normal(loc=0.0, scale=std_dev, size=(X.shape[1], D_FEATURES))
B = np.random.uniform(0, 2 * np.pi, size=D_FEATURES)


### Step 2: Explicit Feature Mapping Transformation
This step successfully transforms the base input data points into the bounded low-dimensional feature space $z(x) = \sqrt{rac{2}{D}} \cos(W^T x + B)$. It explicitly evaluates the sinusoidal geometric mapping defined precisely at the end of **Algorithm 1**, computing empirical variance lower bounds to accurately track the kernel inner product $k(x, y)$.

In [3]:
def transform_rff(X_data, weights, biases, D):
    # Element-wise linear projection and broadcast addition
    projection = np.dot(X_data, weights) + biases
    # Apply cosine activation and scale correctly
    return np.sqrt(2.0 / D) * np.cos(projection)

# Obtain the transformed training and testing features Z
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_SEED)

Z_train = transform_rff(X_train_raw, W, B, D_FEATURES)
Z_test = transform_rff(X_test_raw, W, B, D_FEATURES)


### Step 3: Fast Linear Machine Validation
Instead of expensively computing calculations over a dense $N \times N$ dataset Gram matrix like an explicit non-parametric SVM, we directly evaluate our mapped explicit $D$-dimensional $Z$ features through a linearly scaled parametric model (Ridge Classifier in this testbed). This satisfies the overall proposition made heavily in **Section 1 (Introduction)** to utilize simple $O(D+d)$ scaled learning techniques rather than $O(N^3)$ support vector decomposition bottlenecks.

In [4]:
linear_model = RidgeClassifier(alpha=1.0)
linear_model.fit(Z_train, y_train)

# Evaluate final parametric predictions on testing split
y_pred = linear_model.predict(Z_test)
rff_accuracy = accuracy_score(y_test, y_pred)

print(f"Random Fourier Features (RFF) + Ridge Classifier Accuracy: {rff_accuracy:.4f}")


Random Fourier Features (RFF) + Ridge Classifier Accuracy: 0.9917
